## Expected Paper Metrics

The paper reports the following synthetic-test perception metrics.

| Checkpoint | Description | final mAP@0.50 | final mAP@0.50:0.95 | final macro F1 |
| --- | --- | ---: | ---: | ---: |
| `models/model_base.pt` | trained before synthetic-domain fine-tuning | 0.7442 | 0.6124 | 0.7666 |
| `models/model_finetuned.pt` | fine-tuned on the synthetic dataset | 0.9591 | 0.6450 | 0.9400 |

The notebook computes the same metric family used in the paper:

- three-class detection metrics: `Road-defect-general`, `Person`, `Car`;
- final five-class metrics: `Crocodile Crack`, `Single Crack`, `Pothole`, `Person`, `Car`;
- subtype metrics on road-defect ROIs.

## Runtime Choices

Run this notebook from either:

- a local clone of `EdwinTSalcedo/RDMO-DigitalTwin`; or
- Google Colab after cloning the repository and downloading the datasets from the Google Drive link in the README.

For Colab, a GPU runtime is recommended for fine-tuning. Evaluation of the provided checkpoints also works on CPU, but is slower.

In [1]:
# Global switches. Keep both False for a quick artifact-verification run.
# Set RUN_SYNTHETIC_FINE_TUNING=True only when you want to recreate the final checkpoint.
RUN_BASE_EVALUATION = True
RUN_FINETUNED_EVALUATION = True
RUN_SYNTHETIC_FINE_TUNING = False

# Set to True if you want a few annotated images written under runs/paper_reproduction/.
SAVE_EXAMPLE_PREDICTIONS = False

# Evaluation settings used by the paper workflow.
IMAGE_SIZE = 640
BATCH_SIZE = 8
CONFIDENCE = 0.25
MATCH_IOU = 0.50
SUBSET_SEED = 42

## 1. Install Dependencies and Locate the Repository

The public repository contains the data, checkpoints, Unity project, paper, and this notebook. The reusable training/evaluation package is pulled from the companion experiment-code repository when it is not already present locally.

If you copy the `experiments/` package into this repository later, this notebook will automatically use that local copy instead.

In [2]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def run(command, *, cwd=None, check=True):
    """Run a shell command with visible output for reproducibility."""
    print('$', ' '.join(str(part) for part in command))
    result = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout)
    if check and result.returncode:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result

# Resolve the RDMO-DigitalTwin repository root. In Colab we clone it if needed.
REPO_URL = 'https://github.com/EdwinTSalcedo/RDMO-DigitalTwin.git'
DEFAULT_COLAB_REPO_ROOT = Path('/content/RDMO-DigitalTwin')

if IN_COLAB:
    if not DEFAULT_COLAB_REPO_ROOT.exists():
        run(['git', 'clone', '--depth', '1', REPO_URL, DEFAULT_COLAB_REPO_ROOT])
    REPO_ROOT = DEFAULT_COLAB_REPO_ROOT
else:
    # If the notebook is opened from notebooks/, use the parent repo directory.
    cwd = Path.cwd().resolve()
    REPO_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

os.chdir(REPO_ROOT)
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Repository root:', REPO_ROOT)

Python: 3.13.9
Platform: macOS-26.5.1-arm64-arm-64bit-Mach-O
Repository root: /Users/qp251956/Documents/RDMO-DigitalTwin


In [3]:
# Install runtime dependencies. If your environment is already prepared, set INSTALL_DEPS=False.
INSTALL_DEPS = True

if INSTALL_DEPS:
    # Ultralytics is pinned because the model implementation hooks Detect-head feature tensors.
    packages = [
        'torch',
        'torchvision',
        'ultralytics==8.4.76',
        'opencv-python',
        'numpy',
        'pandas',
        'matplotlib',
        'tqdm',
        'pyyaml',
    ]
    run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--prefer-binary', *packages])

$ /opt/anaconda3/bin/python -m pip install --disable-pip-version-check --prefer-binary torch torchvision ultralytics==8.4.76 opencv-python numpy pandas matplotlib tqdm pyyaml
  Using cached ultralytics-8.4.76-py3-none-any.whl.metadata (41 kB)
  Using cached polars-1.42.1-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_ml_py-13.610.43-py3-none-any.whl.metadata (9.7 kB)
  Using cached ultralytics_thop-2.0.20-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.42.1-cp310-abi3-macosx_11_0_arm64.whl.metadata (1.5 kB)
Using cached ultralytics-8.4.76-py3-none-any.whl (1.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/88.0 MB ? eta -:--:--
   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/88.0 MB 19.9 MB/s eta 0:00:05
   ━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/88.0 MB 19.6 MB/s eta 0:00:05
   ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/88.0 MB 19.4 MB/s eta 0:00:04
   ━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/88.0 MB 19.0 MB/s eta 0:00:04
   ━━━━━━━━╸━━━

In [4]:
# Locate or fetch the experiment package used for model loading, training, and metrics.
# The data/checkpoints still come from RDMO-DigitalTwin; this code package supplies the metric logic.
EXPERIMENT_CODE_URL = 'https://github.com/EdwinTSalcedo/rdmo-simulator.git'
EXPERIMENT_CODE_DIR = Path('/content/rdmo-simulator') if IN_COLAB else (REPO_ROOT.parent / 'rdmo-simulator')

candidate_code_roots = [
    REPO_ROOT,                         # future-proof: local experiments/ package inside RDMO-DigitalTwin
    EXPERIMENT_CODE_DIR,               # sibling local clone or Colab clone
]

CODE_ROOT = next(
    (root for root in candidate_code_roots if (root / 'experiments' / 'common' / 'model.py').is_file()),
    None,
)

if CODE_ROOT is None:
    run(['git', 'clone', '--depth', '1', EXPERIMENT_CODE_URL, EXPERIMENT_CODE_DIR])
    CODE_ROOT = EXPERIMENT_CODE_DIR

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print('Experiment code root:', CODE_ROOT)

# Sanity check imports before continuing.
import torch
import torchvision
import ultralytics
from experiments.workflow import ExperimentConfig, MultiTaskExperiment
from experiments.synthetic_evaluation import print_split_summary
from experiments.synthetic_finetune import SyntheticFineTuneConfig, fine_tune_on_synthetic_dataset
from experiments.synthetic_evaluation import SyntheticEvaluationConfig

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('ultralytics:', ultralytics.__version__)

Experiment code root: /Users/qp251956/Documents/rdmo-simulator
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/qp251956/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
torch: 2.12.1
torchvision: 0.27.1
ultralytics: 8.4.76


## 2. Resolve Data and Checkpoints

Expected repository layout:

```text
RDMO-DigitalTwin/
|-- data/
|   |-- augmented_dataset/
|   `-- synthetic_dataset/
|-- models/
|   |-- model_base.pt
|   `-- model_finetuned.pt
`-- notebooks/
    `-- reproduce_paper_results.ipynb
```

If `data/` is not present because the repository was cloned from GitHub without the datasets, download the datasets from the Google Drive folder linked in the README and extract them under the repository root.

In [5]:
GOOGLE_DRIVE_DATA_URL = 'https://drive.google.com/drive/folders/1bfLm6uia9jM-xPxxl2PxLrq3OVG0Z8TZ?usp=sharing'

# Local override examples:
#   macOS local author copy: /Users/qp251956/Documents/RDMO-DigitalTwin/data
#   Colab Drive shortcut:   /content/drive/MyDrive/RDMO-DigitalTwin/data
DATA_ROOT = Path(os.environ.get('RDMO_DATA_ROOT', REPO_ROOT / 'data')).expanduser().resolve()

if IN_COLAB and not DATA_ROOT.exists():
    # Optional convenience: mount Drive if a user has added the shared data folder as a shortcut.
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_candidates = [
            Path('/content/drive/MyDrive/RDMO-DigitalTwin/data'),
            Path('/content/drive/MyDrive/rdmo/data'),
            Path('/content/drive/MyDrive/data'),
        ]
        DATA_ROOT = next((path for path in drive_candidates if path.exists()), DATA_ROOT)
    except Exception as error:
        print('Drive mount/data lookup skipped:', error)

print('Data root:', DATA_ROOT)
print('Google Drive data folder:', GOOGLE_DRIVE_DATA_URL)

Data root: /Users/qp251956/Documents/RDMO-DigitalTwin/data
Google Drive data folder: https://drive.google.com/drive/folders/1bfLm6uia9jM-xPxxl2PxLrq3OVG0Z8TZ?usp=sharing


In [6]:
import zipfile

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}


def is_yolo_dataset_root(path: Path) -> bool:
    return all((path / split / 'images').is_dir() and (path / split / 'labels').is_dir() for split in ('train', 'valid', 'test'))


def extract_zip_if_needed(zip_path: Path, destination: Path) -> None:
    if destination.exists() and is_yolo_dataset_root(destination):
        return
    if not zip_path.is_file():
        return
    print(f'Extracting {zip_path} -> {destination}')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        target_root = destination.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != target_root and target_root not in target.parents:
                raise ValueError(f'Unsafe path in zip archive: {member.filename}')
        archive.extractall(destination)


def find_yolo_root(start: Path, preferred_name: str) -> Path:
    direct = start / preferred_name
    if is_yolo_dataset_root(direct):
        return direct

    # If only a zip is present, extract it under data/.
    extract_zip_if_needed(start / f'{preferred_name}.zip', direct)
    if is_yolo_dataset_root(direct):
        return direct

    # Some archives include an extra top-level folder. Search below data/ as a fallback.
    if start.exists():
        roots = [path.parent for path in start.rglob('train') if is_yolo_dataset_root(path.parent)]
        matching = [root for root in roots if preferred_name.lower() in str(root).lower()]
        if matching:
            return sorted(matching, key=lambda path: str(path))[0]

    raise FileNotFoundError(
        f'Could not find {preferred_name!r} as a YOLO dataset under {start}.\n'
        f'Download/extract the datasets from: {GOOGLE_DRIVE_DATA_URL}'
    )

AUGMENTED_DATASET_ROOT = find_yolo_root(DATA_ROOT, 'augmented_dataset')
SYNTHETIC_DATASET_ROOT = find_yolo_root(DATA_ROOT, 'synthetic_dataset')

print('Augmented dataset:', AUGMENTED_DATASET_ROOT)
print_split_summary(AUGMENTED_DATASET_ROOT)
print('\nSynthetic dataset:', SYNTHETIC_DATASET_ROOT)
print_split_summary(SYNTHETIC_DATASET_ROOT)

Augmented dataset: /Users/qp251956/Documents/RDMO-DigitalTwin/data/augmented_dataset
train: 32303 images, 32303 labels
valid:  9179 images,  9179 labels
 test:  4693 images,  4693 labels
Split percentages:
  train: 69.96%
  valid: 19.88%
   test: 10.16%

Synthetic dataset: /Users/qp251956/Documents/RDMO-DigitalTwin/data/synthetic_dataset
train:  1568 images,  1568 labels
valid:   444 images,   444 labels
 test:   223 images,   223 labels
Split percentages:
  train: 70.16%
  valid: 19.87%
   test:  9.98%


In [7]:
BASE_CHECKPOINT = Path(os.environ.get('RDMO_BASE_CHECKPOINT', REPO_ROOT / 'models' / 'model_base.pt')).resolve()
FINETUNED_CHECKPOINT = Path(os.environ.get('RDMO_FINETUNED_CHECKPOINT', REPO_ROOT / 'models' / 'model_finetuned.pt')).resolve()
RUNS_DIR = Path(os.environ.get('RDMO_RUNS_DIR', REPO_ROOT / 'runs' / 'paper_reproduction')).resolve()
RUNS_DIR.mkdir(parents=True, exist_ok=True)

for checkpoint in (BASE_CHECKPOINT, FINETUNED_CHECKPOINT):
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint}')
    print(f'{checkpoint.name}: {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)')

print('Outputs will be written to:', RUNS_DIR)

model_base.pt: /Users/qp251956/Documents/RDMO-DigitalTwin/models/model_base.pt (12.5 MB)
model_finetuned.pt: /Users/qp251956/Documents/RDMO-DigitalTwin/models/model_finetuned.pt (12.5 MB)
Outputs will be written to: /Users/qp251956/Documents/RDMO-DigitalTwin/runs/paper_reproduction


## 3. Inspect Checkpoint Metadata

The checkpoints store their training metadata. This is useful for confirming that the model files match the paper workflow before running a full evaluation.

In [8]:
import torch


def checkpoint_summary(path: Path) -> dict:
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    metadata = checkpoint.get('metadata', {})
    history = metadata.get('history', []) if isinstance(metadata, dict) else []
    return {
        'file': path.name,
        'detector_weights': checkpoint.get('detector_weights'),
        'image_size': checkpoint.get('image_size'),
        'roi_size': checkpoint.get('roi_size'),
        'lambda_subtype': checkpoint.get('lambda_subtype'),
        'detection_names': checkpoint.get('detection_names'),
        'subtype_names': checkpoint.get('subtype_names'),
        'best_epoch': metadata.get('epoch') if isinstance(metadata, dict) else None,
        'selection_metric': metadata.get('selection_metric') if isinstance(metadata, dict) else None,
        'selection_value': metadata.get('selection_value') if isinstance(metadata, dict) else None,
        'history_records': len(history),
        'last_history_record': history[-1] if history else None,
    }

for summary in [checkpoint_summary(BASE_CHECKPOINT), checkpoint_summary(FINETUNED_CHECKPOINT)]:
    print(json.dumps(summary, indent=2, default=str))

{
  "file": "model_base.pt",
  "detector_weights": "yolov8n.pt",
  "image_size": 640,
  "roi_size": 7,
  "lambda_subtype": 1.0,
  "detection_names": [
    "Road-defect-general",
    "Person",
    "Car"
  ],
  "subtype_names": [
    "Crocodile Crack",
    "Single Crack",
    "Pothole"
  ],
  "best_epoch": 7,
  "selection_metric": "final_5class_mAP50",
  "selection_value": 0.7430589783541995,
  "history_records": 7,
  "last_history_record": {
    "loss": 30.583804554369777,
    "detection_loss": 30.371410320600468,
    "subtype_loss": 0.21239423497608934,
    "epoch": 7.0,
    "learning_rate": 1e-05,
    "final_5class_mAP50": 0.7430589783541995,
    "final_5class_mAP50_95": 0.6120810692001323,
    "detection_mAP50": 0.8799915750803408,
    "detection_mAP50_95": 0.7916168202619863,
    "subtype_macro_F1": 0.9175693989067278,
    "subtype_accuracy": 0.918297931417465,
    "final_5class_macro_F1": 0.7652923861006335,
    "final_5class_precision": 0.7697810358678207,
    "final_5class_recall

## 4. Evaluation Helpers

This section evaluates a checkpoint on `data/synthetic_dataset/test` and writes the full report to `runs/paper_reproduction/<run_name>/metrics/evaluation.json`.

In [9]:
import numpy as np


def json_safe(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def evaluate_checkpoint(checkpoint: Path, run_name: str, *, include_predictions: bool = False):
    """Evaluate one multitask checkpoint on the synthetic test split."""
    output_dir = RUNS_DIR / run_name
    config = ExperimentConfig(
        dataset_root=SYNTHETIC_DATASET_ROOT,
        detector_weights='yolov8n.pt',
        output_dir=output_dir,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        epochs=1,
        confidence=CONFIDENCE,
        match_iou=MATCH_IOU,
        dataset_fraction=1.0,
        subset_seed=SUBSET_SEED,
        log_every_validation_images=25,
        save_evaluation_predictions=include_predictions,
    )
    experiment = MultiTaskExperiment(config)
    metadata = experiment.model.load_checkpoint(checkpoint)
    report = experiment.evaluate(use_best_checkpoint=False, include_predictions=include_predictions)
    return report, metadata, experiment


def compact_metrics(report: dict) -> dict:
    return {
        'final_mAP50': report['final_5_class']['map50'],
        'final_mAP50_95': report['final_5_class']['map50_95'],
        'final_macro_F1': report['final_5_class']['macro_f1'],
        'detection_mAP50': report['detection']['map50'],
        'detection_mAP50_95': report['detection']['map50_95'],
        'detection_macro_F1': report['detection']['macro_f1'],
        'subtype_gt_macro_F1': report['subtype_ground_truth_rois']['macro_f1'],
        'subtype_pred_macro_F1': report['subtype_predicted_rois']['macro_f1'],
    }

## 5. Evaluate the Published Baseline Checkpoint

This checkpoint represents the multitask YOLOv8n model before synthetic-domain fine-tuning.

In [10]:
base_report = None
base_metadata = None
base_experiment = None

if RUN_BASE_EVALUATION:
    base_report, base_metadata, base_experiment = evaluate_checkpoint(
        BASE_CHECKPOINT,
        'evaluate_model_base_on_synthetic_test',
        include_predictions=False,
    )
    print('Baseline compact metrics:')
    print(json.dumps(compact_metrics(base_report), indent=2))
else:
    print('RUN_BASE_EVALUATION=False; skipped baseline evaluation.')

Adapting pretrained detector from 80 to 3 classes.
Overriding model.yaml nc=80 with nc=3
Transferred 319/355 items from pretrained weights


Final test evaluation:   0%|          | 0/223 [00:00<?, ?image/s]

Computing detection and final mAP...
Metric aggregation completed in 0.2s
Test inference and metrics completed in 24.0s

PRIMARY final_5class_mAP50=0.1166 mAP50_95=0.0369

detection
  Road-defect-general      P=0.4496 R=0.3948 F1=0.4204
  Person                   P=0.0000 R=0.0000 F1=0.0000
  Car                      P=0.6667 R=0.0077 F1=0.0152
  macro P=0.3721 R=0.1342 F1=0.1452 mAP50=0.0973 mAP50_95=0.0283
  confusion matrix (last row/column = background):
[[486   1   0 744]
 [ 40   0   0 833]
 [  1   0   4 515]
 [554 213   2   0]]

final_5_class
  Crocodile Crack          P=0.3462 R=0.0885 F1=0.1410
  Single Crack             P=0.3333 R=0.2452 F1=0.2826
  Pothole                  P=0.4144 R=0.5405 F1=0.4691
  Person                   P=0.0000 R=0.0000 F1=0.0000
  Car                      P=0.6667 R=0.0077 F1=0.0152
  macro P=0.3521 R=0.1764 F1=0.1816 mAP50=0.1166 mAP50_95=0.0369
  confusion matrix (last row/column = background):
[[ 27  34  43   0   0 201]
 [  1 115   0   0   0 353]


## 6. Evaluate the Published Fine-Tuned Checkpoint

This is the final checkpoint used for the perception results in the paper.

In [ ]:
finetuned_report = None
finetuned_metadata = None
finetuned_experiment = None

if RUN_FINETUNED_EVALUATION:
    finetuned_report, finetuned_metadata, finetuned_experiment = evaluate_checkpoint(
        FINETUNED_CHECKPOINT,
        'evaluate_model_finetuned_on_synthetic_test',
        include_predictions=SAVE_EXAMPLE_PREDICTIONS,
    )
    print('Fine-tuned compact metrics:')
    print(json.dumps(compact_metrics(finetuned_report), indent=2))
else:
    print('RUN_FINETUNED_EVALUATION=False; skipped fine-tuned evaluation.')

## 7. Compare Against the Paper Values

Small numerical differences can occur across hardware, PyTorch versions, and Ultralytics internals. Evaluating the provided checkpoints should be very close to the paper values.

In [ ]:
PAPER_TARGETS = {
    'model_base.pt': {
        'final_mAP50': 0.7442,
        'final_mAP50_95': 0.6124,
        'final_macro_F1': 0.7666,
    },
    'model_finetuned.pt': {
        'final_mAP50': 0.9591,
        'final_mAP50_95': 0.6450,
        'final_macro_F1': 0.9400,
        'detection_mAP50': 0.9738,
        'subtype_pred_macro_F1': 0.9973,
    },
}

rows = []
for name, report in [('model_base.pt', base_report), ('model_finetuned.pt', finetuned_report)]:
    if report is None:
        continue
    observed = compact_metrics(report)
    for metric, expected in PAPER_TARGETS[name].items():
        value = observed.get(metric)
        rows.append({
            'checkpoint': name,
            'metric': metric,
            'paper': expected,
            'observed': value,
            'delta': None if value is None else value - expected,
        })

try:
    import pandas as pd
    comparison = pd.DataFrame(rows)
    display(comparison.round(4))
except Exception:
    print(json.dumps(rows, indent=2))

summary_path = RUNS_DIR / 'paper_metric_comparison.json'
summary_path.write_text(json.dumps(json_safe(rows), indent=2), encoding='utf-8')
print('Saved comparison:', summary_path)

## 8. Optional: Recreate the Final Synthetic Fine-Tuning Run

Leave `RUN_SYNTHETIC_FINE_TUNING=False` for normal review. Set it to `True` in the first cell when you want to fine-tune from `models/model_base.pt` using the full synthetic train/valid/test splits.

Configuration used here:

- starting checkpoint: `models/model_base.pt`;
- dataset: `data/synthetic_dataset`;
- epochs: 10;
- learning rate: `2e-5`;
- weight decay: `1e-4`;
- early stopping patience: 5;
- checkpoint selection metric: final five-class mAP@0.50 on the synthetic validation split.

In [ ]:
fine_tune_result = None

if RUN_SYNTHETIC_FINE_TUNING:
    base_config = SyntheticEvaluationConfig(
        dataset_archive=DATA_ROOT / 'synthetic_dataset.zip',
        experiments_root=RUNS_DIR,
        repo_root=REPO_ROOT,
        dataset_cache_dir=DATA_ROOT,
        local_dataset_archive=DATA_ROOT / 'synthetic_dataset.zip',
        train_run_name='shared_backbone_multitask_v1',
        eval_run_name='synthetic_finetune_yolov8n_eval',
        detector_weights='yolov8n.pt',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        epochs=15,
        learning_rate=1e-4,
        lambda_subtype=1.0,
        train_dataset_fraction=1.0,
        eval_dataset_fraction=1.0,
        subset_seed=SUBSET_SEED,
        confidence=CONFIDENCE,
        match_iou=MATCH_IOU,
        log_every_batches=5,
        log_every_validation_images=25,
        save_evaluation_predictions=False,
    )

    fine_tune_config = SyntheticFineTuneConfig(
        base=base_config,
        output_dir=RUNS_DIR / 'synthetic_finetune_yolov8n',
        epochs=10,
        batch_size=BATCH_SIZE,
        learning_rate=2e-5,
        weight_decay=1e-4,
        lambda_subtype=1.0,
        train_dataset_fraction=1.0,
        validation_dataset_fraction=1.0,
        test_dataset_fraction=1.0,
        workers=2,
        early_stopping_patience=5,
        lr_decay_patience=3,
        lr_decay_factor=0.5,
        min_learning_rate=1e-7,
    )

    fine_tune_result = fine_tune_on_synthetic_dataset(
        fine_tune_config,
        dataset_root=SYNTHETIC_DATASET_ROOT,
        checkpoint=BASE_CHECKPOINT,
        evaluate_after_training=True,
    )

    print('Fine-tuning output:', fine_tune_result['output_dir'])
    print('Best checkpoint:', fine_tune_result['best_checkpoint'])
    print('Test metrics from recreated checkpoint:')
    print(json.dumps(compact_metrics(fine_tune_result['report']), indent=2))
else:
    print('RUN_SYNTHETIC_FINE_TUNING=False; skipped training.')

## 9. Optional: Visualize Final Predictions

Run this after evaluating `model_finetuned.pt`. The images are written under `runs/paper_reproduction/evaluate_model_finetuned_on_synthetic_test/artifacts/`.

In [ ]:
if SAVE_EXAMPLE_PREDICTIONS and finetuned_experiment is not None:
    sample_indices = [0, len(finetuned_experiment.test_dataset) // 2, len(finetuned_experiment.test_dataset) - 1]
    for sample_index in sample_indices:
        print(f'Sample {sample_index}')
        finetuned_experiment.show_prediction(sample_index=sample_index)
else:
    print('Set SAVE_EXAMPLE_PREDICTIONS=True and run the fine-tuned evaluation first to create visual examples.')

## 10. Output Files

The notebook writes reproducibility artifacts under:

```text
runs/paper_reproduction/
|-- evaluate_model_base_on_synthetic_test/
|   `-- metrics/evaluation.json
|-- evaluate_model_finetuned_on_synthetic_test/
|   `-- metrics/evaluation.json
|-- synthetic_finetune_yolov8n/              # only if RUN_SYNTHETIC_FINE_TUNING=True
|   |-- checkpoints/best.pt
|   |-- checkpoints/last.pt
|   `-- metrics/evaluation.json
`-- paper_metric_comparison.json
```

These files are intentionally placed under `runs/`, which is ignored by the repository. They are local reproducibility outputs, not source files.